# 실험 04: 에러 자가 치유 (Auto-fixing Output Parsers)

## 1. 실험 제목과 목표
- **제목**: 실무에서 필수적인 에러 복구 파이프라인 구축
- **목표**: 서비스가 죽지 않도록, `OutputFixingParser`와 `RetryOutputParser`를 사용하여 망가진 JSON 응답을 모델 스스로 복구하게 만드는 Fault Tolerance(장애 허용) 기법을 마스터합니다.

## 2. 실험 개요
1. **실무 실험 1**: 괄호나 따옴표가 빠져서 고장 난 가짜 JSON 문자열을 `OutputFixingParser`에 통과시켜 강제로 꿰매는 마법 시연
2. **실무 실험 2**: 단순히 포맷이 망가진 게 아니라 내용물 자체가 규칙에 맞지 않을 때, 원래의 질문과 에러 내용을 모델에게 다시 던지며 혼내서(Retry) 제대로 된 답변을 받아오는 고급 기법

## 3. 라이브러리 import

In [3]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser, RetryOutputParser

## 4. 환경 변수 로드 및 모델 준비

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

## 5. 실무 실험 1: OutputFixingParser (오타 고치기)
문법적으로 살짝 고장난 텍스트를 LLM의 지능을 빌려 문법에 맞게 강제 변환합니다.

In [5]:
class Actor(BaseModel):
    name: str = Field(description="배우의 이름")
    filmography: list[str] = Field(description="출연한 영화 목록")

parser = PydanticOutputParser(pydantic_object=Actor)

# 괄호가 제대로 안 닫히고, 따옴표 규칙도 엉망인 고장난 JSON
misformatted_json = "{'name': '송강호', 'filmography': ['기생충', '괴물', '택시운전사'"

print("=== [1. 일반 파서로 시도하면?] ===")
try:
    parser.parse(misformatted_json)
except Exception as e:
    print("🔥 에러 발생! JSON 디코딩 실패:", e)

print("\n=== [2. OutputFixingParser 요정 투입] ===")
# LLM을 활용해 파서를 수리하는 새로운 파서를 만듭니다.
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

# 다시 시도!
fixed_actor = fixing_parser.parse(misformatted_json)
print("타입:", type(fixed_actor))
print("내용:", fixed_actor)
print("-> 결과: LLM이 망가진 텍스트를 스윽 읽어보고, 닫히지 않은 괄호와 따옴표를 완벽한 Pydantic 객체로 복원(Self-healing)해 냈습니다.")

=== [1. 일반 파서로 시도하면?] ===
🔥 에러 발생! JSON 디코딩 실패: Invalid json output: {'name': '송강호', 'filmography': ['기생충', '괴물', '택시운전사'
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

=== [2. OutputFixingParser 요정 투입] ===
타입: <class '__main__.Actor'>
내용: name='송강호' filmography=['기생충', '괴물', '택시운전사']
-> 결과: LLM이 망가진 텍스트를 스윽 읽어보고, 닫히지 않은 괄호와 따옴표를 완벽한 Pydantic 객체로 복원(Self-healing)해 냈습니다.


## 6. 실무 실험 2: RetryOutputParser (논리적 오류 재시도)
단순 오타가 아니라, 정보 자체가 부족하거나 프롬프트를 무시한 경우, "왜 틀렸는지" 알려주고 다시 시키는 기법입니다.

In [6]:
# LLM이 실수로 'filmography' 항목 전체를 빼먹은 심각한 상황 시뮬레이션
bad_response = """
{
  "name": "마동석"
}
"""

# 원래 사용자가 했던 질문 (재시도할 때 맥락을 알아야 하므로 필요함)
original_prompt_value = PromptTemplate.from_template("{actor}에 대해 알려줘.").invoke({"actor": "마동석"})

print("=== [1. FixingParser로 시도하면?] ===")
try:
    # FixingParser는 텍스트 포맷만 고칠 줄 알지, 아예 없는 정보(영화 목록)를 지어내지는 못합니다.
    fixing_parser.parse(bad_response)
except Exception as e:
    print("🔥 에러 발생! 필수 값이 누락됨:", type(e).__name__)

print("\n=== [2. RetryOutputParser 선생님 투입] ===")
# LLM에게 원본 질문, 틀린 응답, 그리고 파서 에러 메시지를 통째로 던져서 다시 생성시킵니다.
retry_parser = RetryOutputParser.from_llm(parser=parser, llm=llm)

retried_actor = retry_parser.parse_with_prompt(bad_response, original_prompt_value)
print("타입:", type(retried_actor))
print("내용:", retried_actor)
print("-> 결과: 원래 질문이 '마동석에 대해 알려줘'였다는 것을 인지한 LLM이, 누락되었던 영화 목록을 스스로 검색(생성)해서 완벽히 채워왔습니다!")

=== [1. FixingParser로 시도하면?] ===

=== [2. RetryOutputParser 선생님 투입] ===


OutputParserException: Invalid json output: 마동석은 대한민국의 배우이자 프로듀서로, 1971년 3월 22일에 태어났습니다. 본명은 마동석이며, 영어 이름은 'Don Lee'입니다. 그는 주로 액션과 코미디 장르의 영화에서 활발히 활동하고 있으며, 특히 '부산행', '신과함께' 시리즈, '범죄도시' 등에서의 역할로 유명합니다. 마동석은 강한 체격과 카리스마 있는 연기로 많은 사랑을 받고 있으며, 한국뿐만 아니라 해외에서도 인지도를 높이고 있습니다. 또한, 그는 헐리우드 영화에도 출연하며 국제적인 배우로 자리매김하고 있습니다.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

## 7. 결과 해석

1. **실무 파이프라인의 완성**: 아무리 프롬프트를 잘 짜고 `with_structured_output`을 써도, 인터넷 통신 오류나 오픈소스 LLM의 한계 등으로 언젠간 파싱이 깨집니다. 이 방어 로직들이 서버 다운을 막아주는 생명줄이 됩니다.
2. **비용(Cost) 문제**: Fixing이나 Retry는 백그라운드에서 LLM API를 '한 번 더' 호출하는 구조입니다. 즉, 안정성을 얻는 대신 비용과 시간이 2배로 들 수 있으므로 남용하지 말고 `try-except` 블록 안의 최후의 수단으로만 써야 합니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `OutputFixingParser`는 괄호 하나 빠진 정도의 문법 오류(Syntax Error)를 기가 막히게 잘 고침.
* 하지만 아예 내용이 빠져버린 논리 오류(Semantic Error)는 못 고치는데, 이때 `RetryOutputParser`가 원래 질문을 다시 보고 정답을 새로 써오는 것을 확인.

### 다음 개선 방향
* 이제 LLM 기초, LangChain 체인 조립, 파싱, 그리고 에러 핸들링까지 모두 마스터했음!
* LLM이 모르는 최신 정보나 회사 내부 문서를 LLM에게 주입하는 **RAG (Retrieval-Augmented Generation)** 기초 챕터로 넘어갈 완벽한 준비가 됨.